Cell 1: Install Dependencies

In [ ]:
!pip install -q transformers datasets sentencepiece evaluate accelerate torch scikit-learn sacrebleu rouge-score

!fusermount -u /content/drive    # Unmount if already mounted
!rm -rf /content/drive           # Remove leftover files/folder
from google.colab import drive
drive.mount('/content/drive')    # Mount again

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 10.2 MB/s eta 0:00:00
fusermount: failed to unmount /content/drive: No such file or directory
Mounted at /content/drive


Cell 2: Mount Drive & Imports

In [ ]:
import pandas as pd
import torch
import numpy as np
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    MBartForConditionalGeneration,
    MBart50TokenizerFast,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
import evaluate
import warnings
warnings.filterwarnings('ignore')

print(f"GPU Available: {torch.cuda.is_available()}")
print(f"GPU Name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

GPU Available: True
GPU Name: Tesla T4


Cell 3: Load and Prepare Data

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset

# Define the file path
csv_path = "/content/drive/MyDrive/Final Data Set - 10000 Data set.csv"

try:
    # Load only the main 10k dataset
    # on_bad_lines='skip' ensures that rows with extra commas don't crash the script
    df = pd.read_csv(csv_path, on_bad_lines='skip')
    print(f"Successfully loaded file: {csv_path}")
except Exception as e:
    print(f"Error loading CSV: {e}")
    # Fallback: try reading with a different encoding if needed
    df = pd.read_csv(csv_path, on_bad_lines='skip', encoding='utf-8')

# Clean the data
df = df.dropna().drop_duplicates()
print(f"Total samples after cleaning: {len(df)}")

# Format the data for mBART
def format_example(row):
    return {
        "input_text": f"Explain the word '{row['english_word']}' in Sinhala.\nContext: {row['context_sentence']}",
        "target_text": row['sinhala_explanation']
    }

# Apply formatting
formatted_data = df.apply(format_example, axis=1, result_type="expand")

# Split into Training and Validation (20% for validation)
train_df, valid_df = train_test_split(formatted_data, test_size=0.2, random_state=42)

# Convert to Hugging Face Dataset format
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
valid_dataset = Dataset.from_pandas(valid_df.reset_index(drop=True))

print(f"Train: {len(train_dataset)}, Validation: {len(valid_dataset)}")

Successfully loaded file: /content/drive/MyDrive/Final Data Set - 10000 Data set.csv
Total samples after cleaning: 8105
Train: 6484, Validation: 1621


Cell 4: Load Model and Tokenizer

In [ ]:
model_name = "facebook/mbart-large-50-many-to-many-mmt"

tokenizer = MBart50TokenizerFast.from_pretrained(model_name)
model = MBartForConditionalGeneration.from_pretrained(model_name)

tokenizer.src_lang = "en_XX"
tokenizer.tgt_lang = "si_LK"

print(f"Model loaded: {sum(p.numel() for p in model.parameters())/1e6:.1f}M parameters")

tokenizer_config.json:   0%|          | 0.00/529 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

Model loaded: 610.9M parameters


Cell 5: Tokenize Data

In [ ]:
# CHANGE 3: Increased max lengths to capture full explanations
def tokenize_function(examples):
    model_inputs = tokenizer(
        examples["input_text"],
        max_length=128,  # Reduced input length
        truncation=True,
        padding=False
    )
    labels = tokenizer(
        examples["target_text"],
        max_length=256,  # Keep full target length
        truncation=True,
        padding=False
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Tokenizing...")
tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=train_dataset.column_names)
tokenized_valid = valid_dataset.map(tokenize_function, batched=True, remove_columns=valid_dataset.column_names)
print("Done!")

Tokenizing...


Map:   0%|          | 0/6484 [00:00<?, ? examples/s]

Map:   0%|          | 0/1621 [00:00<?, ? examples/s]

Done!


Cell 6: Setup Metrics

In [ ]:
bleu_metric = evaluate.load("sacrebleu")
rouge_metric = evaluate.load("rouge")

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    if isinstance(preds, tuple):
        preds = preds[0]

    # Convert logits to token IDs
    if np.issubdtype(preds.dtype, np.floating):
        preds = np.argmax(preds, axis=-1)

    # CRITICAL FIX: Replace -100 with pad_token_id so the tokenizer doesn't crash
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]

    bleu_result = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels)
    rouge_result = rouge_metric.compute(predictions=decoded_preds, references=[ref[0] for ref in decoded_labels])

    return {
        "bleu": bleu_result["score"],
        "rouge1": rouge_result["rouge1"],
        "rouge2": rouge_result["rouge2"],
        "rougeL": rouge_result["rougeL"],
    }

Cell 7: IMPROVED Training Setup

In [ ]:
output_dir = "/content/drive/MyDrive/sinhala-model-v2"

training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5, # Lower rate for stability
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    warmup_steps=200,
    weight_decay=0.01,
    logging_steps=50,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    greater_is_better=True,
    predict_with_generate=True,
    generation_max_length=256,
    generation_num_beams=4,

    # STABILITY FIXES
    fp16=False,         # Standard precision is safer for large datasets
    max_grad_norm=1.0,  # Prevents exploding gradients

    report_to="none",
    dataloader_pin_memory=False,
)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True)

# This line defines the 'trainer' variable
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Trainer is now defined and ready!")

Trainer is now defined and ready!


Cell 8: Train Model

In [ ]:
print("Starting training with 10 epochs...")
try:
    train_result = trainer.train()

    # Save model locally first, then copy to drive
    local_model_path = "/content/local_model"
    trainer.save_model(local_model_path)
    print(f"\nModel saved locally at: {local_model_path}")

    # Copy to Google Drive
    import shutil
    shutil.copytree(local_model_path, output_dir, dirs_exist_ok=True)
    print(f"Model copied to Drive: {output_dir}")

    print(f"\nTraining complete! Final loss: {train_result.training_loss:.4f}")

    eval_results = trainer.evaluate()
    print(f"\nFinal Metrics:")
    print(f"  BLEU: {eval_results['eval_bleu']:.2f}")
    print(f"  ROUGE-L: {eval_results['eval_rougeL']:.4f}")

except Exception as e:
    print(f"Error during training: {e}")
    print("Attempting to save current model state...")
    try:
        local_model_path = "/content/local_model"
        trainer.save_model(local_model_path)
        print(f"Model saved locally at: {local_model_path}")
    except:
        print("Could not save model")

Starting training with 10 epochs...


Epoch,Training Loss,Validation Loss,Bleu,Rouge1,Rouge2,Rougel
1,2.032800,1.932105,7.504836,0.963798,0.001645,0.964168
2,1.713900,1.782883,8.540617,0.971191,0.001645,0.971489
3,1.447200,1.724558,9.663679,0.974090,0.001234,0.974604
4,1.200500,1.707269,10.194748,0.973802,0.001851,0.974193
5,1.088700,1.699019,11.454237,0.971869,0.001234,0.972200
6,0.904200,1.707801,11.740463,0.971520,0.001851,0.971828
7,0.832200,1.723820,12.499739,0.968641,0.001234,0.969151
8,0.707300,1.733335,12.319265,0.969669,0.001234,0.970286


Epoch,Training Loss,Validation Loss,Bleu,Rouge1,Rouge2,Rougel
1,2.032800,1.932105,7.504836,0.963798,0.001645,0.964168
2,1.713900,1.782883,8.540617,0.971191,0.001645,0.971489
3,1.447200,1.724558,9.663679,0.974090,0.001234,0.974604
4,1.200500,1.707269,10.194748,0.973802,0.001851,0.974193
5,1.088700,1.699019,11.454237,0.971869,0.001234,0.972200
6,0.904200,1.707801,11.740463,0.971520,0.001851,0.971828
7,0.832200,1.723820,12.499739,0.968641,0.001234,0.969151
8,0.707300,1.733335,12.319265,0.969669,0.001234,0.970286
9,0.678100,1.742470,12.437585,0.971211,0.001234,0.971520
10,0.621400,1.747484,12.450906,0.972075,0.001234,0.972435


There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].



Model saved locally at: /content/local_model
Model copied to Drive: /content/drive/MyDrive/sinhala-model-v2

Training complete! Final loss: 1.1640



Final Metrics:
  BLEU: 12.50
  ROUGE-L: 0.9692


Cell 9: Simple Inference Testing Function

In [ ]:
import re

def test_model_simple(word, context):
    """Simple test function for inference with forced formatting"""
    input_text = f"Explain the word '{word}' in Sinhala.\nContext: {context}"
    inputs = tokenizer(input_text, return_tensors="pt", max_length=128, truncation=True)

    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}
        model.cuda()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.lang_code_to_id["si_LK"],
            max_length=256,
            min_length=15,
            num_beams=4,
            repetition_penalty=2.5,
            length_penalty=1.2,
            no_repeat_ngram_size=3,
            early_stopping=True,
            temperature=0.7,
        )

    prediction = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

    # --- FORMATTING LOGIC ---

    # 1. Clean up any existing English words in brackets (e.g., "(Wrench)")
    prediction = re.sub(r'\([a-zA-Z\s]+\)', '', prediction).strip()

    # 2. Check if it already starts with "Word යනු"
    prefix = f"{word} යනු "
    if not prediction.startswith(word):
        # If the model didn't include the word, add it manually
        # Also check if it started with "යනු" and remove it to avoid "Word යනු යනු"
        prediction = re.sub(r'^යනු\s*', '', prediction)
        prediction = f"{prefix}{prediction}"
    elif f"{word} " in prediction and "යනු" not in prediction:
        # If the word is there but "යනු" is missing
        prediction = prediction.replace(f"{word} ", prefix, 1)

    return prediction

Cell 10: Real-Time Inference Testing

In [ ]:
import time

def test_with_timing(word, context):
    """Test model with timing"""
    start_time = time.time()
    prediction = test_model_simple(word, context)
    response_time = time.time() - start_time
    return prediction, response_time

# Test cases
print("\n" + "="*70)
print("REAL-TIME MODEL TESTING")
print("="*70)

test_cases = [
    ("beautiful", "The sunset was absolutely beautiful."),
    ("intelligence", "Artificial intelligence is rapidly advancing."),
    ("friendship", "True friendship lasts forever."),
    ("courage", "It takes courage to stand up for what is right."),
    ("happy", "She felt very happy today."),
    ("freedom", "Freedom is important for everyone."),
    ("knowledge", "Knowledge is power in modern society."),
    ("peace", "World peace is a noble goal."),
    ("science","Modern science is typically divided into two."),
    ("School","School is organized in several different organizational models"),
    ("compulsory","which is sometimes compulsory."),
]

total_time = 0
successful_tests = 0

for i, (word, context) in enumerate(test_cases, 1):
    try:
        prediction, response_time = test_with_timing(word, context)
        total_time += response_time
        successful_tests += 1

        print(f"\n[Test {i}] Word: '{word}'")
        print(f"Context: {context}")
        print(f"Explanation: {prediction}")
        print(f"⏱️  Response Time: {response_time:.2f}s ({response_time*1000:.0f}ms)")
        print("-"*70)
    except Exception as e:
        print(f"\n[Test {i}] ❌ Failed for word '{word}': {e}")
        print("-"*70)

# Summary
if successful_tests > 0:
    avg_time = total_time / successful_tests
    print(f"\n📊 Summary:")
    print(f"  Total tests: {len(test_cases)}")
    print(f"  Successful: {successful_tests}")
    print(f"  Failed: {len(test_cases) - successful_tests}")
    print(f"  Average response time: {avg_time:.2f}s")
    print(f"  Total time: {total_time:.2f}s")
    print(f"  Status: {'✅ Working properly' if avg_time < 5 else '⚠️  Slow responses'}")
else:
    print("\n❌ All tests failed. Please check the model.")
print("="*70)


REAL-TIME MODEL TESTING

[Test 1] Word: 'beautiful'
Context: The sunset was absolutely beautiful.
Explanation: ඉතා ලස්සනට හෝ ආකර්ෂණීය ලෙස පෙනෙන දෙයකි (සුඛෝපභෝගී).
⏱️  Response Time: 0.80s (803ms)
----------------------------------------------------------------------

[Test 2] Word: 'intelligence'
Context: Artificial intelligence is rapidly advancing.
Explanation: බුද්ධිමය බුද්ධිය හෝ බුද්ධිමය හැකියාව (ඉවද් ය බුද්ධිය) යි.
⏱️  Response Time: 0.68s (680ms)
----------------------------------------------------------------------

[Test 3] Word: 'friendship'
Context: True friendship lasts forever.
Explanation: මිත් රත්වය යනු අන් අය අතර ඇති දැඩි සම්බන්ධතාවයයි (පෙම්වතාත්වය).
⏱️  Response Time: 0.47s (467ms)
----------------------------------------------------------------------

[Test 4] Word: 'courage'
Context: It takes courage to stand up for what is right.
Explanation: නිර්භීත බව හෝ බියක් වෙනුවෙන් නැගී සිටීමේ ශක්තියයි (ධෛර්යය).
⏱️  Response Time: 0.55s (550ms)
--------------------------------

Cell 11: Test on Validation Samples

In [ ]:
print("\n" + "="*70)
print("VALIDATION SAMPLE PREDICTIONS")
print("="*70)

# Test on first 5 validation samples
validation_samples = valid_dataset.select(range(min(10, len(valid_dataset))))

for i, sample in enumerate(validation_samples):
    input_parts = sample['input_text'].split('\n')
    word = input_parts[0].replace("Explain the word '", "").replace("' in Sinhala.", "")
    context = input_parts[1].replace('Context: ', '')

    prediction = test_model_simple(word, context)
    ground_truth = sample['target_text']

    print(f"\n[Sample {i+1}] Word: {word}")
    print(f"Context: {context}")
    print(f"\nGround Truth ({len(ground_truth.split())} words):")
    print(f"  {ground_truth}")
    print(f"\nPrediction ({len(prediction.split())} words):")
    print(f"  {prediction}")
    print("-"*70)

print("\n✅ Testing complete!")


VALIDATION SAMPLE PREDICTIONS

[Sample 1] Word: Nook
Context: She sat in a quiet nook reading a book.

Ground Truth (11 words):
  Nook යනු කාමරයක හෝ ස්ථානයක ඇති කුඩා හුදකලා මුල්ලක් හෝ අස්සකි.

Prediction (7 words):
  Nook යනු කුඩා ගුහාවක හෝ අවට ස්ථානයකි.
----------------------------------------------------------------------

[Sample 2] Word: Irate
Context: The irate customer demanded a refund.

Ground Truth (5 words):
  Irate යනු දැඩි කෝපයෙන් පසුවන.

Prediction (9 words):
  Irate යනු ඉතා කේන්තියෙන් සහ නොසන්සුන්ව සිටින (මාර්ගයක්) ස්වභාවයයි.
----------------------------------------------------------------------

[Sample 3] Word: Vivid
Context: He gave a vivid description of the event.

Ground Truth (9 words):
  Vivid යනු ඉතා පැහැදිලි දීප්තිමත් හෝ සිතේ මැවෙන සුළු.

Prediction (8 words):
  Vivid යනු ඉතා පැහැදිලි සහ සෘජුව පෙනෙන  දෙයකි.
----------------------------------------------------------------------

[Sample 4] Word: Citadel
Context: The king hid in the citadel during the attack.

Gr

New Word Testing Interface

In [ ]:
import re

def predict_unseen(word, context):
    # Prepare input
    input_text = f"Explain the word '{word}' in Sinhala.\nContext: {context}"
    inputs = tokenizer(input_text, return_tensors="pt", max_length=128, truncation=True)

    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}

    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.lang_code_to_id["si_LK"],
            max_length=256,
            num_beams=4,
            repetition_penalty=2.5,
            length_penalty=1.0,
            early_stopping=True
        )

    prediction = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

    # Clean up: Remove extra labels and brackets
    prediction = prediction.split("හේතුව:")[0].split("යථාර්ථය:")[0]
    prediction = re.sub(r'\s*\([^)]*\)', '', prediction).strip()

    # Force the "Word යනු" format
    prefix = f"{word} යනු "
    if not prediction.startswith(word):
        prediction = re.sub(r'^යනු\s*', '', prediction)
        prediction = f"{prefix}{prediction}"

    return prediction

# --- TEST AREA ---
# You can change these to any word you think isn't in your dataset!
test_word = "problem"
test_context = "There is also the difficulty of knowledge acquisition, the problem of obtaining knowledge for AI applications."

result = predict_unseen(test_word, test_context)

print(f"WORD: {test_word}")
print(f"CONTEXT: {test_context}")
print(f"\nMODEL EXPLANATION:\n{result}")

NameError: name 'tokenizer' is not defined